# Environment-holdout molecular study — results

Informed vs blind `GeoScalarGNNMolecular` under the 5 × 5 chemotype × environment cross-validation (design in the
README, *Environment-Holdout Molecular Study*), on the AIMEl test sets and deployed on the QM9 remainder. Produces the
same-molecule figure of the main text and the informed-vs-blind figure, tables and error-vs-distance figure of the SI.
The first section needs only the committed tables; the rest needs the run outputs next to each `config.json` and
writes its tables to `env_holdout_results/`.

**Statistics.** The fold is the unit (n = 25): every fold has its own test set and the within-repeat ICC of the fold
metrics is ≤ 0 (§0). Fold means ± 95 % t-intervals. Informed vs blind: Tukey HSD among the eight models of a property,
as in the atomic comparison; the tables also give Holm-adjusted paired t and Wilcoxon p.

In [ ]:
import importlib
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = os.path.abspath(os.path.join('..', '..'))
for _p in (os.path.join(project_root, 'analysis', 'atomic'), os.path.join(project_root, 'analysis', 'molecular')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import env_holdout_analysis as eha
import env_holdout_figures as ehf
import compare_atomic_envs_6b as cae6
for _m in (eha, ehf, cae6):
    importlib.reload(_m)
ra = ehf.ra

%matplotlib inline
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 80)
FIG_DIR = os.path.join('figures', 'env_holdout')
os.makedirs(FIG_DIR, exist_ok=True)
RUNS_DIR, OUT_DIR = eha.RUNS_DIR, eha.OUT_DIR

## Figures from the shipped tables

The same-molecule figure, the error-vs-distance figure and the fold design, from the committed CSVs and
`config.json` files. Everything after this section needs `test_preds.pkl` and `remainder_preds.pkl` (README,
*Environment-Holdout Molecular Study*, steps 2-3).


In [ ]:
# ── style and text, shared by the three figures of this notebook (edit for the paper) ──
FONTS = {'title': 18, 'label': 16, 'tick': 12, 'legend': 12, 'star': 18}   # QT-Net's published sizes (analyse_inference)
TITLE_WEIGHT, LABEL_WEIGHT = 'normal', 'normal'
VARIANT_LABELS = {'informed': 'Informed', 'blind': 'Blind'}
VARIANT_COLORS = {'informed': '#d62728', 'blind': '#1f77b4'}
XLABEL = 'Training fraction (%)'
XTICK_FMT = '{p:g}'     # ticks 1, 5, 10, 100: '0.05' and '0.1' collide in narrow panels
FACTOR_COLORS = {'environments': '#b40426', 'chemotype': '#3b4cc0'}    # coolwarm's ends, as in the novelty figures
FACTOR_LABELS = {'environments': 'Environments', 'chemotype': 'Chemotype'}   # 'unseen' is in the y label
SET_STYLES = {'test': ('-', ''), 'remainder': ('--', '')}              # (line style, marker) per test set; '' = no marker

def fig_novelty(summary):
    ehf.plot_novelty_groups(summary, value='rel', layout='2x2',          # layout: '2x2' or '1x4'
                            sets=(('test', 'AIMEl test'), ('remainder', 'QM9 remainder')),
                            variant_labels=VARIANT_LABELS,
                            group_colors={'0': '#3b4cc0', '1–2': '#f39c7d', '≥3': '#b40426'},
                            group_labels={'0': 'No rare atom', '1–2': '1–2 rare atoms', '≥3': '≥3 rare atoms'},
                            title_fmt='{variant} — {set}',
                            ylabel='Error relative to molecules\nwith no rare atom', xlabel=XLABEL,
                            fonts={'title': 17, 'label': 16, 'tick': 14, 'legend': 14},   # ~7-8.5 pt printed 5.5 in wide
                            title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT, legend_loc='upper left',
                            sharey=True, xtick_fmt=XTICK_FMT, figsize=(10.8, 8.4), save_path=f'{FIG_DIR}/error_vs_novelty.png')
    plt.show()


def fig_increase(table):
    ehf.plot_matched_increase(table, level='factor', variants=('blind', 'informed'),     # left to right
                              sets=(('test', 'AIMEl test'), ('remainder', 'QM9 remainder')),
                              variant_labels=VARIANT_LABELS, factor_colors=FACTOR_COLORS, factor_labels=FACTOR_LABELS,
                              set_styles=SET_STYLES, dodge=0.035,
                              title_fmt='',                        # no titles: the caption says left blind, right informed
                              split_legend=True,                   # colours (factors) left, line styles (test sets) right
                              set_legend_labels={'test': 'AIMEl test sets', 'remainder': 'QM9 remainder'},
                              legend_handlelength=3.0,             # long enough to show the dashes
                              ylabel='Error increase when\nunseen in training (%)', xlabel=XLABEL,
                              fonts={'title': 16, 'label': 15, 'tick': 13, 'legend': 13},   # main text, ~7-9 pt printed 5.5 in wide
                              title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT, legend_loc='upper left',
                              set_legend_loc='upper left', xtick_fmt=XTICK_FMT, figsize=(10, 3.5),
                              save_path=f'{FIG_DIR}/same_molecule_increase.png')
    plt.show()


In [ ]:
fig_increase(pd.read_csv(os.path.join(OUT_DIR, 'same_molecule_increase_factor.csv')))
fig_novelty(pd.read_csv(os.path.join(OUT_DIR, 'novelty_groups.csv')))
display(ehf.fold_design(RUNS_DIR))


In [ ]:
import glob

assert glob.glob(f'{RUNS_DIR}/*/fold_*/frac_*/test_preds.pkl'), \
    'Everything below needs test_preds.pkl and remainder_preds.pkl, written by steps 2-3 of the README.'


## 0. Checks

Complete grid, unseen environments of panels B/C against each run's `train_ids.json`, recomputed tables against the
committed CSVs, and the design of the 25 folds.

In [ ]:
inv = ehf.run_inventory(RUNS_DIR)
display(ehf.check_inventory(inv))
labels = cae6.load_env_labels()
print('% of test molecules with >= 1 environment absent from the run\'s training molecules:')
display(ehf.coherence_table(ehf.env_support(labels, RUNS_DIR)))

scales = ehf.load_scales(OUT_DIR)
committed_fm = pd.read_csv(os.path.join(OUT_DIR, 'aimel_fold_metrics.csv'))
fm = eha.aimel_fold_metrics(RUNS_DIR)
diff = ehf.compare_tables(fm, committed_fm, ['method', 'cv_cycle', 'panel', 'property'])
print(f"recomputed aimel_fold_metrics == committed CSV (max relative difference {diff['max_rel_diff'].max():.1e})")
icc = ra.assess_fold_correlation(fm, models=ehf.MODELS, panel_col='panel', panels=['A', 'B', 'C', 'D', 'all'],
                                 properties=ehf.PROPS, metric='R2', split='test')
print(f"within-repeat ICC of fold R2: median {icc['ICC'].median():.2f}, max {icc['ICC'].max():.2f}")

In [ ]:
# the design, read from the runs: one row per fold, and how often each AIMEl molecule is scored in each panel
display(ehf.fold_design(RUNS_DIR))
visits, visits_by_k = ehf.panel_visits(labels, cae6.load_env_folds(), RUNS_DIR)
print('Number of B visits (columns) against k, the number of environment folds the molecule carries (rows):')
display(visits_by_k)
print(f"in A or B exactly 5 times: {100 * ((visits['A'] + visits['B']) == 5).mean():.1f} % | "
      f"B = k: {100 * (visits['B'] == visits['k']).mean():.1f} % | C = 4k: {100 * (visits['C'] == 4 * visits['k']).mean():.1f} % | "
      f"D visits: {visits['D'].min()}–{visits['D'].max()}")

## 1. R² of informed vs blind

AIMEl test sets (top) and QM9 remainder (bottom); stars: Tukey HSD p < 0.05, in the colour of the better variant.

In [ ]:
r2_aimel = ehf.informed_vs_blind(ehf.aimel_metrics(fm, 'all'), verdict_by='p_tukey')        # verdict: Tukey HSD p < 0.05
r2_rem = ehf.informed_vs_blind(ehf.remainder_metrics_all(RUNS_DIR), verdict_by='p_tukey')
ehf.plot_r2_curves({'aimel': r2_aimel, 'remainder': r2_rem},
                   row_labels={'aimel': 'AIMEl test', 'remainder': 'QM9 remainder'},
                   prop_titles={'alpha': r'$\alpha$', 'gap': r'$\Delta\varepsilon$', 'U0': r'$U_0$', 'Cv': r'$C_v$'},
                   ylabel=r'$R^2$', xlabel=XLABEL, variant_labels=VARIANT_LABELS, variant_colors=VARIANT_COLORS,
                   fonts={'title': 19, 'label': 17, 'tick': 15, 'legend': 15, 'star': 20},   # ~7-9 pt printed 5.5 in wide
                   title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT, legend_loc='lower right', xtick_fmt=XTICK_FMT,
                   figsize=(12, 6.6), save_path=f'{FIG_DIR}/r2_informed_vs_blind.png')
plt.show()

**Tables.** All statistics, then the LaTeX tables of the SI (Tukey p only).

In [ ]:
print('AIMEl test set (panels A-D pooled)')
display(ehf.readable(r2_aimel))
print('QM9 remainder')
display(ehf.readable(r2_rem))

In [ ]:
print(ehf.latex_ivb_table(r2_aimel, label='tab:env_holdout_r2_aimel', holm=False,
      caption=r'$R^2$ on the AIMEl test sets (fold mean $\pm$ 95\% CI half-width, $n = 25$). $p_\mathrm{Tukey}$: '
              r'Tukey HSD between the two variants among the eight models of a target (two variants $\times$ four '
              r'fractions). Bold: better variant ($p_\mathrm{Tukey} < 0.05$).'))
print()
print(ehf.latex_ivb_table(r2_rem, label='tab:env_holdout_r2_remainder', holm=False,
      caption=r'As Table~\ref{tab:env_holdout_r2_aimel}, on the 98\,096 QM9 molecules outside AIMEl.'))

## 2. Panel R² relative to the whole test set

Panels hold different molecules; §5 measures each factor on the same molecules.

In [ ]:
panel_folds, panel_tab = ehf.panel_delta(fm)
ehf.plot_panel_delta(panel_tab,
                     panel_titles={'A': 'A\nChemotype held out\nEnvironments seen',
                                   'B': 'B\nChemotype held out\nEnvironments unseen',
                                   'C': 'C\nChemotype seen\nEnvironments unseen',
                                   'D': 'D\nChemotype seen\nEnvironments seen'},
                     ylabel='$R^2_\\mathrm{panel} - R^2_\\mathrm{test}$\n(mean over properties)', xlabel=XLABEL,
                     variant_labels=VARIANT_LABELS, variant_colors=VARIANT_COLORS,
                     fonts={'title': 17, 'label': 17, 'tick': 15, 'legend': 15, 'star': 20},   # ~7-8 pt printed 5.5 in wide
                     title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT, legend_loc='lower left', xtick_fmt=XTICK_FMT,
                     figsize=(12, 4.8), save_path=f'{FIG_DIR}/panel_delta_r2.png')
plt.show()
ehf.readable(panel_tab).sort_values(['panel', 'fraction'])

**Panel vs panel.** B − A and C − D: unseen environments; A − D and B − C: unseen chemotype.

In [ ]:
contrasts = ehf.panel_contrasts(fm)
_c = ehf.readable(contrasts, value_cols=('diff',))
display(_c.pivot_table(index=['contrast', 'variant'], columns='fraction', values='diff', aggfunc='first')
          .loc[list(ehf.PANEL_CONTRASTS)])
print('Holm-adjusted p (paired t), max over the table:', ehf.fmt_p(contrasts['p_t_holm'].max()))

## 3. Error against the distance to the model's training set

Distance of an atom: to its 15th-nearest training atom of the same element in the unscaled 6-Bohr PCA space; rare
beyond the p99 of the model's validation atoms. Molecules are grouped by their number of rare atoms (0 / 1–2 / ≥ 3).

In [ ]:
pca = cae6.soap_pca_6b()
dist = ehf.distance_tables(pca, labels, RUNS_DIR, n_remainder=20000)
errors = eha.molecule_errors(RUNS_DIR, scales=scales)
rem_ids = ehf.remainder_sample(labels, 20000)
on = ['fold', 'fraction', 'index']
test_tab = ehf.wide_errors(errors).merge(dist[dist['set'] == 'test'].drop(columns=['set', 'panel']), on=on,
                                         how='left', validate='many_to_one')
rem_tab = ehf.remainder_sample_errors(RUNS_DIR, rem_ids, scales).merge(
    dist[dist['set'] == 'remainder'].drop(columns=['set', 'panel']), on=on, how='left', validate='many_to_one')
assert test_tab['n_far_15'].notna().all() and rem_tab['n_far_15'].notna().all()
tab = pd.concat([test_tab.assign(set='test'), rem_tab.assign(set='remainder')], ignore_index=True)
val = dist[dist['set'] == 'val']
print('chance level — validation molecules with >= 1 rare atom:',
      {f: f'{100 * v:.1f} %' for f, v in val.groupby('fraction')['n_far_15'].apply(lambda s: (s > 0).mean()).items()})
nov_summary, nov_contrast = ehf.novelty_groups(tab, 'n_far_15')

fig_novelty(nov_summary)


**Tables.** Group shares and errors, relative to the molecules without rare atoms.

In [ ]:
_s = nov_summary.assign(cell=[f'{sh:.1f} % | {ehf.fmt_ci(a, b, c)} | ×{r:.2f}' for sh, a, b, c, r in
                              zip(nov_summary['share_%'], nov_summary['nerr'], nov_summary['nerr_lo'],
                                  nov_summary['nerr_hi'], nov_summary['rel'])])
print('share of molecules | normalised error [95 % CI] | relative to no-rare')
display(_s.pivot_table(index=['set', 'variant', 'fraction'], columns='group', values='cell', aggfunc='first', observed=True))
nov_contrast.assign(diff=[ehf.fmt_ci(a, b, c) for a, b, c in zip(nov_contrast['diff'], nov_contrast['diff_lo'], nov_contrast['diff_hi'])],
                    p_t_holm=nov_contrast['p_t_holm'].map(ehf.fmt_p))[
    ['set', 'variant', 'fraction', 'nerr_0', 'nerr_3plus', 'ratio', 'diff', 'p_t_holm']].round(3)

**Within molecule.** Spearman ρ between a remainder molecule's error and its distance, both centred over the 25 models of a fraction.

In [ ]:
within = ehf.summarise_folds(ehf.within_molecule(rem_tab, 'D_max_15'), ['variant', 'fraction'], 'rho_within')
within.assign(rho=[ehf.fmt_ci(a, b, c) for a, b, c in zip(within['mean'], within['ci_lo'], within['ci_hi'])],
              p_t=within['p_t'].map(ehf.fmt_p))[['variant', 'fraction', 'n_folds', 'rho', 'p_t']]

## 4. Share of the error from zwitterions

In [ ]:
frame = cae6.load_frame()
zw_ids = frame.loc[cae6.formal_charges(frame['smiles'].tolist())['n_charged_atoms'].to_numpy() > 0, 'index']
zw_runs = ehf.zwitterion_effect(zw_ids, RUNS_DIR)
print(f"{len(zw_ids)} zwitterions in QM9; per fold: {zw_runs.groupby('set')['n_zw'].mean().round(1).to_dict()}; "
      f"share of molecules (%): {zw_runs.groupby('set')['zw_%_of_molecules'].mean().round(2).to_dict()}")
print('share of the total absolute error (%)')
display(ehf.zwitterion_summary(zw_runs, 'zw_%_of_abs_err').round(2))
print('ΔR² = R² without zwitterions − R² with all molecules')
display(ehf.zwitterion_summary(zw_runs, 'dR2').round(4))

## 5. The same molecules with and without each factor

Per fold k and contrast X | Y (X: factor unseen, Y: seen): Δ = 100 · (Σᵢ e_ik / Σᵢ ē_iY − 1), averaged over the four
targets, where e_ik is molecule i's error scaled by the target's standard deviation over AIMEl and ē_iY its mean error
in the folds that place it in Y. On the remainder, the 25 models of a fraction play the folds.

In [ ]:
# error increase on the same molecules: per fold, then fold mean ± 95 % t-interval
per_fold = pd.concat([ehf.matched_increase_folds(errors),                         # AIMEl test sets
                      ehf.remainder_matched_increase_folds(RUNS_DIR, scales)],     # QM9 remainder: every model, every molecule
                     ignore_index=True)
increase = ehf.summarise_increase(per_fold)                                        # the four contrasts
increase_factor = ehf.summarise_increase(ehf.factor_increase_folds(per_fold))      # one value per factor
_order = [c[0] for c in ehf.MATCHED_CONTRASTS]
_cell = lambda t: t.assign(cell=[ehf.fmt_ci(a, b, c, 1) for a, b, c in zip(t['mean'], t['ci_lo'], t['ci_hi'])])
print('Error increase on the same molecules, % (fold mean [95 % CI]), one value per factor:')
display(_cell(increase_factor).pivot_table(index=['set', 'variant', 'fraction'], columns='contrast', values='cell',
                                           aggfunc='first')[['environments', 'chemotype']])
_ff = ehf.factor_increase_folds(per_fold).pivot_table(index=['set', 'variant', 'fraction', 'fold'], columns='contrast',
                                                     values='increase_%')
env_vs_chem = ehf.summarise_increase((_ff['environments'] - _ff['chemotype']).rename('increase_%').reset_index()
                                     .assign(contrast='environments − chemotype'))
env_vs_chem['p_t_holm'] = ehf.holm(env_vs_chem['p_t'])
print('Environments − chemotype, paired by fold (percentage points; one-sample t, Holm over the 16 cells):')
display(_cell(env_vs_chem).assign(p_t_holm=env_vs_chem['p_t_holm'].map(ehf.fmt_p))
        .set_index(['set', 'variant', 'fraction'])[['n_folds', 'cell', 'p_t_holm']])
print('Per contrast (fold mean, %):')
display(increase.pivot_table(index=['set', 'variant', 'fraction'], columns='contrast', values='mean')[_order].round(1))
by_prop = ehf.summarise_folds(ehf.matched_increase_folds(errors, by_property=True),
                              ['variant', 'fraction', 'contrast', 'property'], 'increase_%')
for _f, _g in by_prop.groupby(by_prop['contrast'].str.split(' ').str[0]):
    print(f"AIMEl test, {_f}: positive in {int((_g['mean'] > 0).sum())} of {len(_g)} variant × fraction × property × "
          f"contrast cells ({_g['mean'].min():.1f}–{_g['mean'].max():.1f} %), fold-level 95 % CI above 0 in "
          f"{int((_g['ci_lo'] > 0).sum())}")
matched = ehf.matched_contrasts(errors)
print('AIMEl test, different molecules (panel means, %), for comparison:')
display(matched.pivot_table(index=['variant', 'fraction'], columns='contrast', values='unmatched_%')[_order].round(1))
desc = ehf.descriptor_effect(errors)
print('Descriptors on the same molecules: informed vs blind, % of blind error (AIMEl test)')
display(desc.pivot_table(index='fraction', columns='property', values='informed_vs_blind_%').round(1))
print('Per panel, relative to the pooled test set (blind, frac 1.0, fold means)')
display(ehf.panel_variance(RUNS_DIR).pivot_table(index='panel', columns='property',
                                                 values=['variance_ratio', 'mse_ratio']).round(2))

**Figure (main text).** The mean of each factor's two contrasts; left blind, right informed.

In [ ]:
fig_increase(increase_factor)


**The four contrasts separately.**

In [ ]:
CONTRAST_LABELS = {'environment | chemotype held out': 'Environments, chemotype held out (B − A)',
                   'environment | chemotype seen': 'Environments, chemotype seen (C − D)',
                   'chemotype | environments seen': 'Chemotype, environments seen (A − D)',
                   'chemotype | environments unseen': 'Chemotype, environments unseen (B − C)'}
ehf.plot_matched_increase(increase, level='contrast', variants=('blind', 'informed'),
                          sets=(('test', 'AIMEl test'), ('remainder', 'QM9 remainder')),
                          variant_labels=VARIANT_LABELS, factor_colors=FACTOR_COLORS, contrast_labels=CONTRAST_LABELS,
                          dodge=0.035, title_fmt='{variant} — {set}', ylabel='Error increase when\nunseen in training (%)',
                          xlabel=XLABEL, fonts={'title': 17, 'label': 16, 'tick': 14, 'legend': 13},   # ~7-8 pt printed 5.5 in wide
                          title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT,
                          legend_loc='upper left', xtick_fmt=XTICK_FMT, figsize=(11, 8.8),
                          save_path=f'{FIG_DIR}/same_molecule_increase_contrasts.png')
plt.show()

**Error against distance, within molecule.**

In [ ]:
curve = ehf.within_molecule_curve(tab, 'D_mean_15', n_bins=10)
SET_COLORS = {'test': ['#7fb9da', '#3b8bc2', '#135fa7', '#08306b'],         # 'Blues', AIMEl as in the t-SNE figures
              'remainder': ['#fb7a5a', '#e83429', '#b61319', '#67000d']}    # 'Reds', QM9 remainder as in the t-SNE figures
ehf.plot_within_molecule_by_variant(curve, variants=('blind', 'informed'),     # left to right
                                    sets=(('test', 'AIMEl test'), ('remainder', 'QM9 remainder')),
                                    variant_labels=VARIANT_LABELS, set_colors=SET_COLORS,
                                    set_styles={'test': '-', 'remainder': '--'},
                                    ci='bar', dodge=0.004,                     # or ci='band'
                                    title_fmt='{variant}',
                                    xlabel="Distance to the model's training environments,\nvs the molecule's average (Δ log)",
                                    ylabel="Error vs the molecule's\naverage over models (%)",
                                    legend_fmt='{set}, training fraction',
                                    fonts={'title': 17, 'label': 15, 'tick': 14, 'legend': 13},   # ~7-9 pt printed 5.5 in wide
                                    title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT, legend_loc='upper left',
                                    figsize=(10.5, 4.8),
                                    save_path=f'{FIG_DIR}/error_vs_distance_within_molecule.png')
plt.show()
_ends = curve.sort_values('x').groupby(['set', 'variant', 'fraction'])
display(pd.DataFrame({'x_nearest_bin': _ends['x'].first(), 'x_farthest_bin': _ends['x'].last(),
                      'error_nearest_bin_%': _ends['y'].first(), 'error_farthest_bin_%': _ends['y'].last()}).round(2))
within_mean = ehf.summarise_folds(ehf.within_molecule(rem_tab, 'D_mean_15'), ['variant', 'fraction'], 'rho_within')
print('Remainder, within-molecule Spearman ρ of error and mean distance (fold-level):')
within_mean.assign(rho=[ehf.fmt_ci(a, b, c) for a, b, c in zip(within_mean['mean'], within_mean['ci_lo'], within_mean['ci_hi'])],
                   p_t=within_mean['p_t'].map(ehf.fmt_p))[['variant', 'fraction', 'n_folds', 'rho', 'p_t']]

In [ ]:
# the tables of this notebook, next to eha's
for name, t in {'r2_informed_vs_blind_aimel': r2_aimel, 'r2_informed_vs_blind_remainder': r2_rem,
                'panel_delta_r2': panel_tab, 'panel_contrasts_r2': contrasts, 'novelty_groups': nov_summary, 'novelty_groups_contrast': nov_contrast,
                'novelty_within_molecule': within, 'matched_contrasts': matched,
                'same_molecule_increase': increase, 'same_molecule_increase_factor': increase_factor,
                'same_molecule_increase_per_fold': per_fold,
                'same_molecule_env_vs_chemotype': env_vs_chem, 'descriptor_effect_matched': desc,
                'within_molecule_curve': curve}.items():
    t.to_csv(os.path.join(OUT_DIR, f'{name}.csv'), index=False)
print('wrote', len(os.listdir(OUT_DIR)), 'files in', OUT_DIR)